1. **Title**

Day 16 - Student Wellbeing Statistical Analysis & Probability

This notebook analyzes student wellbeing data using descriptive statistics,
outlier detection, probability, conditional probability, Bayes' theorem,
and the normal distribution.

2. **Import Libraries**

In [1]:
import pandas as pd
import numpy as np

3. **Load Dataset**

In [2]:
df = pd.read_csv("Day16_Student_Wellbeing_Survey.csv")
df.head()

,Student_ID,Age,Faculty,Year_of_Study,City,Accommodation,Scholarship,Part_Time_Job,Internet_Quality,Preferred_Study_Space,Weekly_Study_Hours,Average_Sleep_Hours,Daily_Screen_Time_Hours,Exercise_Days_Per_Week,Commute_Time_Minutes,Stress_Score,Academic_Readiness_Score,Overall_Satisfaction,Monthly_Discretionary_Spending,Social_Activity_Hours_Per_Week
0,STU0001,21,Data Science,3,Bengaluru,Hostel,No,Yes,Good,Café,15.2,5.7,3.2,3,11.8,5.4,63.1,3.2,5131.0,7.8
1,STU0002,21,Data Science,4,Pune,Hostel,Yes,No,Poor,Library,17.6,7.7,3.5,4,20.3,2.1,74.2,3.8,3508.0,9.2
2,STU0003,22,Life Sciences,2,Chandigarh,Home,Yes,No,Poor,Home,18.9,6.4,2.8,1,15.0,5.2,77.3,4.0,4584.0,7.7
3,STU0004,18,Business,4,Bengaluru,Hostel,Yes,Yes,Average,Café,19.5,6.7,3.0,5,12.1,4.7,67.0,4.0,5456.0,5.3
4,STU0005,20,Business,3,Jammu,Shared Apartment,Yes,No,Good,Home,18.4,7.3,3.0,1,30.2,5.4,65.6,3.2,11732.0,15.9


4. **Dataset Overview**

In [3]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 600
Columns: 20


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Student_ID                      600 non-null    object 
 1   Age                             600 non-null    int64  
 2   Faculty                         600 non-null    object 
 3   Year_of_Study                   600 non-null    int64  
 4   City                            600 non-null    object 
 5   Accommodation                   600 non-null    object 
 6   Scholarship                     600 non-null    object 
 7   Part_Time_Job                   600 non-null    object 
 8   Internet_Quality                600 non-null    object 
 9   Preferred_Study_Space           600 non-null    object 
 10  Weekly_Study_Hours              600 non-null    float64
 11  Average_Sleep_Hours             600 non-null    float64
 12  Daily_Screen_Time_Hours         600 

In [5]:
df.isnull().sum()

,0
Student_ID,0
Age,0
Faculty,0
Year_of_Study,0
City,0
Accommodation,0
Scholarship,0
Part_Time_Job,0
Internet_Quality,0
Preferred_Study_Space,0


5. **Descriptive Statistics**

In [6]:
variables = [
    "Weekly_Study_Hours",
    "Average_Sleep_Hours",
    "Daily_Screen_Time_Hours",
    "Stress_Score",
    "Academic_Readiness_Score"
]

In [7]:
stats = pd.DataFrame(index=variables)

stats["Mean"] = df[variables].mean()
stats["Median"] = df[variables].median()
stats["Mode"] = df[variables].mode().iloc[0]

stats

,Mean,Median,Mode
Weekly_Study_Hours,15.691000,15.40,13.1
Average_Sleep_Hours,6.997500,7.00,7.0
Daily_Screen_Time_Hours,4.503000,4.20,3.5
Stress_Score,4.464500,4.50,4.7
Academic_Readiness_Score,71.773167,71.65,71.1


### Interpretation

The mean and median give the central values of each variable, while the mode shows the most frequently occurring value. Comparing the mean and median also gives an idea of whether the data is relatively balanced or influenced by extreme values.

6. **Range, Variance, Standard Deviation, Q1, Q3 and IQR**

In [8]:
stats["Range"] = df[variables].max() - df[variables].min()
stats["Variance"] = df[variables].var()
stats["Standard Deviation"] = df[variables].std()
stats["Q1"] = df[variables].quantile(0.25)
stats["Q3"] = df[variables].quantile(0.75)
stats["IQR"] = stats["Q3"] - stats["Q1"]

stats

,Mean,Median,Mode,Range,Variance,Standard Deviation,Q1,Q3,IQR
Weekly_Study_Hours,15.691000,15.40,13.1,30.0,19.217047,4.383725,13.000,18.425,5.425
Average_Sleep_Hours,6.997500,7.00,7.0,5.0,0.703383,0.838679,6.475,7.600,1.125
Daily_Screen_Time_Hours,4.503000,4.20,3.5,11.2,3.469006,1.862527,3.200,5.400,2.200
Stress_Score,4.464500,4.50,4.7,8.4,2.782494,1.668081,3.300,5.700,2.400
Academic_Readiness_Score,71.773167,71.65,71.1,56.1,93.981967,9.694430,65.200,78.025,12.825


In [9]:
stats["Standard Deviation"].sort_values(ascending=False)

,Standard Deviation
Academic_Readiness_Score,9.694430
Weekly_Study_Hours,4.383725
Daily_Screen_Time_Hours,1.862527
Stress_Score,1.668081
Average_Sleep_Hours,0.838679


### Interpretation

Academic Readiness Score has the largest absolute standard deviation among the selected variables. This means its values show the greatest spread around their mean when measured in their original units.

7. **IQR Outlier Detection**

In [10]:
outlier_variables = [
    "Weekly_Study_Hours",
    "Daily_Screen_Time_Hours",
    "Commute_Time_Minutes",
    "Monthly_Discretionary_Spending"
]

In [11]:
def find_outliers(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[column] < lower) | (df[column] > upper)]

    print(column)
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower Bound:", lower)
    print("Upper Bound:", upper)
    print("Number of Outliers:", len(outliers))
    print()

In [12]:
for column in outlier_variables:
    find_outliers(column)

Weekly_Study_Hours
Q1: 13.0
Q3: 18.424999999999997
IQR: 5.424999999999997
Lower Bound: 4.862500000000004
Upper Bound: 26.562499999999993
Number of Outliers: 8

Daily_Screen_Time_Hours
Q1: 3.2
Q3: 5.4
IQR: 2.2
Lower Bound: -0.10000000000000009
Upper Bound: 8.700000000000001
Number of Outliers: 18

Commute_Time_Minutes
Q1: 13.375
Q3: 31.525
IQR: 18.15
Lower Bound: -13.849999999999998
Upper Bound: 58.75
Number of Outliers: 4

Monthly_Discretionary_Spending
Q1: 4113.0
Q3: 7808.5
IQR: 3695.5
Lower Bound: -1430.25
Upper Bound: 13351.75
Number of Outliers: 20



8. **Compare Mean and Median Before & After Removing Outliers**

In [13]:
column = "Weekly_Study_Hours"

Q1 = df[column].quantile(0.25)
Q3 = df[column].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

clean_data = df[
    df[column].between(lower, upper)
][column]

print("Before removing outliers")
print("Mean:", df[column].mean())
print("Median:", df[column].median())

print("\nAfter removing outliers")
print("Mean:", clean_data.mean())
print("Median:", clean_data.median())

Before removing outliers
Mean: 15.690999999999997
Median: 15.4

After removing outliers
Mean: 15.60287162162162
Median: 15.350000000000001


### Interpretation

After removing the outliers, the mean and median of weekly study hours changed only slightly. This suggests that the detected extreme values had a relatively small effect on the overall centre of the data.

9. **Define Probability Events**

*Probability Events*

A = Student has a part-time job

B = Student has Stress Score ≥ 7

C = Student has a scholarship

D = Student exercises at least 3 days per week

In [14]:
A = df["Part_Time_Job"] == "Yes"
B = df["Stress_Score"] >= 7
C = df["Scholarship"] == "Yes"
D = df["Exercise_Days_Per_Week"] >= 3

10. **Basic Probabilities**

In [15]:
print("P(A) =", A.mean())
print("P(B) =", B.mean())
print("P(C) =", C.mean())
print("P(D) =", D.mean())

P(A) = 0.25333333333333335
P(B) = 0.075
P(C) = 0.30666666666666664
P(D) = 0.5933333333333334


In [16]:
print("P(A):", round(A.mean() * 100, 2), "%")
print("P(B):", round(B.mean() * 100, 2), "%")
print("P(C):", round(C.mean() * 100, 2), "%")
print("P(D):", round(D.mean() * 100, 2), "%")

P(A): 25.33 %
P(B): 7.5 %
P(C): 30.67 %
P(D): 59.33 %


11. **Union and Intersection**

In [17]:
p_A_or_B = (A | B).mean()
print("P(A or B) =", p_A_or_B)

P(A or B) = 0.27666666666666667


In [18]:
p_A_and_B = (A & B).mean()
print("P(A and B) =", p_A_and_B)

P(A and B) = 0.051666666666666666


### Interpretation

About 27.67% of students belong to at least one of the two groups: students with a part-time job or students with high stress. About 5.17% belong to both groups.

12. **Conditional Probability**

In [19]:
p_A_given_B = (A & B).sum() / B.sum()
print("P(A|B) =", p_A_given_B)

P(A|B) = 0.6888888888888889


In [20]:
p_B_given_A = (A & B).sum() / A.sum()
print("P(B|A) =", p_B_given_A)

P(B|A) = 0.20394736842105263


### Interpretation

Among students with high stress, around 68.89% have a part-time job. On the other hand, around 20.39% of students with a part-time job have a stress score of 7 or higher.

13. **Mutually Exclusive Events**

In [21]:
year1 = df["Year_of_Study"] == 1
year4 = df["Year_of_Study"] == 4

print("Students in Year 1:", year1.sum())
print("Students in Year 4:", year4.sum())
print("Intersection:", (year1 & year4).sum())

Students in Year 1: 164
Students in Year 4: 120
Intersection: 0


### Interpretation

Year 1 and Year 4 are mutually exclusive events because a student cannot belong to both years of study at the same time. Their intersection is zero.

14. **Check Whether A and B Are Independent**

In [22]:
p_A = A.mean()
p_B = B.mean()
p_A_and_B = (A & B).mean()

print("P(A and B):", p_A_and_B)
print("P(A) × P(B):", p_A * p_B)

P(A and B): 0.051666666666666666
P(A) × P(B): 0.019


### Interpretation

P(A and B) is different from P(A) × P(B). Therefore, based on this comparison, events A and B do not appear to be independent in this dataset. Having a part-time job appears to be associated with the probability of having high stress.

15. **Bayes' Theorem**

In [23]:
p_A = A.mean()
p_B_given_A = (B & A).sum() / A.sum()

not_A = ~A
p_not_A = not_A.mean()

p_B_given_not_A = (B & not_A).sum() / not_A.sum()

print("P(A) =", p_A)
print("P(B|A) =", p_B_given_A)
print("P(B|not A) =", p_B_given_not_A)
print("P(not A) =", p_not_A)

P(A) = 0.25333333333333335
P(B|A) = 0.20394736842105263
P(B|not A) = 0.03125
P(not A) = 0.7466666666666667


In [24]:
bayes_result = (
    p_B_given_A * p_A
) / (
    p_B_given_A * p_A +
    p_B_given_not_A * p_not_A
)

print("P(A|B) using Bayes =", bayes_result)

P(A|B) using Bayes = 0.6888888888888889


In [25]:
print("Direct P(A|B):", p_A_given_B)
print("Bayes P(A|B):", bayes_result)

Direct P(A|B): 0.6888888888888889
Bayes P(A|B): 0.6888888888888889


### Interpretation

The Bayes theorem result agrees with the direct conditional probability calculation. Both give approximately 0.6889, confirming that the calculation is consistent.

16. **Normal Distribution Analysis**

In [26]:
mean_score = df["Academic_Readiness_Score"].mean()
std_score = df["Academic_Readiness_Score"].std()

print("Mean:", mean_score)
print("Standard Deviation:", std_score)

Mean: 71.77316666666667
Standard Deviation: 9.694429667762527


17. **Highest and Lowest Scores**

In [27]:
highest_score = df["Academic_Readiness_Score"].max()
lowest_score = df["Academic_Readiness_Score"].min()

print("Highest Score:", highest_score)
print("Lowest Score:", lowest_score)

Highest Score: 98.0
Lowest Score: 41.9


In [28]:
z_highest = (highest_score - mean_score) / std_score
z_lowest = (lowest_score - mean_score) / std_score

print("Z-score of highest score:", z_highest)
print("Z-score of lowest score:", z_lowest)

Z-score of highest score: 2.705350828481123
Z-score of lowest score: -3.081477476287823


### Interpretation

The highest Academic Readiness Score has a Z-score of approximately +2.71, meaning it is about 2.71 standard deviations above the average.

The lowest score has a Z-score of approximately -3.08, meaning it is about 3.08 standard deviations below the average. This makes the lowest score unusually low compared with the rest of the dataset.

18. **68-95-99.7 Rule**

*Empirical Rule*

For an approximately normal distribution:

Within 1 standard deviation → 68%

Within 2 standard deviations → 95%

Within 3 standard deviations → 99.7%

In [29]:
print("Within 1 SD:", mean_score - std_score, "to", mean_score + std_score)
print("Within 2 SD:", mean_score - 2*std_score, "to", mean_score + 2*std_score)
print("Within 3 SD:", mean_score - 3*std_score, "to", mean_score + 3*std_score)

Within 1 SD: 62.07873699890414 to 81.4675963344292
Within 2 SD: 52.38430733114161 to 91.16202600219172
Within 3 SD: 42.689877663379086 to 100.85645566995424


### Interpretation

According to the empirical rule, approximately 68% of students are expected to fall within one standard deviation of the mean, about 95% within two standard deviations, and about 99.7% within three standard deviations, assuming the scores follow an approximately normal distribution.

19. **Final 5 Statistical Observations**



- Weekly study hours are centred around 15 to 16 hours, while average sleep is very close to 7 hours per day.

- Academic Readiness Score shows the largest absolute spread among the main variables, with scores ranging from 41.9 to 98.

- Daily screen time has more detected IQR outliers than weekly study hours and commute time, showing that a small group of students spends unusually high amounts of time on screens.

- Students with a part-time job and students with high stress do not appear to be independent events in this dataset, since their joint probability is noticeably higher than the product of their individual probabilities.

- The highest Academic Readiness Score is around 2.71 standard deviations above the mean, while the lowest score is just over 3 standard deviations below the mean, making the lowest score particularly unusual.

20. **Conclusion**

The Student Wellbeing dataset was analyzed using descriptive statistics, IQR-based outlier detection, probability, conditional probability, Bayes' theorem, and the normal distribution.

The analysis provides an overview of student study habits, sleep, screen time, stress, exercise, and academic readiness. The probability analysis also showed a noticeable relationship between part-time employment and high stress, while the Z-score analysis highlighted unusually high and low academic readiness scores.